In [0]:
from pyspark.sql.types import (
    DecimalType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("opportunity_name", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("selling_price", DecimalType(10, 2), True),
    StructField("discount_amount", DecimalType(10, 2), True),  
    StructField("transaction_timestamp", StringType(), True),
    StructField("payment_mode", StringType(), True),
    StructField("sales_channel", StringType(), True),
])


In [0]:

source_path = ("/Volumes/dbr_dev/volumes/blob_source/transactions_source")

checkpoint_path = ("/Volumes/dbr_dev/volumes/blob_source/checkpoints/transactions")

df = (
    spark.readStream  # noqa: F821
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")
    .option("nullValue", "null")
    .schema(schema)
    .load(source_path)
)

query = (
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("dbr_dev.blob_bronze.transactions")
)

query.awaitTermination()